In [22]:
from inspect import getmembers, isfunction
from pathlib import Path
from typing import Callable
import lhotse.recipes

def _get_lhotse_download_dataset_fns() -> tuple[str, Callable]:
    fns = []
    for fn_name, fn in getmembers(lhotse.recipes, isfunction):
        if fn_name.startswith('download_'):
            dataset_name = fn_name.split('_', maxsplit=1)[1]
            fns.append((fn_name, dataset_name, fn))
    return fns

def get_available_datasets_lhotse() -> list[str]:
    return [dataset_name for (fn_name, dataset_name, fn) in _get_lhotse_download_dataset_fns()]

def get_available_datasets() -> list[str]:
    # may add own datasets here if needed someday
    return get_available_datasets_lhotse()


def download_dataset(name: str, target_dir: str|Path = None, force_download: bool = False, prepare: bool = True, **kwargs) -> Path:
    if target_dir is None:
        target_dir = name

    dl_fns = _get_lhotse_download_dataset_fns()
    
    for (fn_name, dataset_name, fn) in dl_fns:
        if dataset_name != name:
            continue
        
        return fn(target_dir=target_dir, force_download=force_download)

    raise ValueError(f"Dataset {name} does not exist.")

print(get_available_datasets())
download_dataset('voxconverse', 'datasets/voxconverse')

['adept', 'aishell', 'aishell3', 'aishell4', 'ali_meeting', 'ami', 'atcosim', 'baker_zh', 'but_reverb_db', 'bvcc', 'chime6', 'cmu_arctic', 'cmu_indic', 'daily_talk', 'dipco', 'earnings21', 'earnings22', 'ears', 'edacc', 'fleurs', 'gigast', 'grid', 'heroico', 'hifitts', 'himia', 'icsi', 'libricss', 'librimix', 'librispeech', 'libritts', 'librittsr', 'ljspeech', 'magicdata', 'mdcc', 'medical', 'mobvoihotwords', 'mtedx', 'musan', 'reazonspeech', 'rir_noise', 'sbcsae', 'spatial_librispeech', 'speechcommands', 'spgispeech', 'stcmds', 'tedlium', 'tedlium2', 'thchs_30', 'this_american_life', 'timit', 'uwb_atcc', 'vctk', 'voxceleb1', 'voxceleb2', 'voxconverse', 'voxpopuli', 'xbmu_amdo31', 'yesno']


TypeError: download_voxconverse() got an unexpected keyword argument 'target_dir'

In [ ]:
def get_lhotse_prepare_dataset(dataset_name: str) -> Callable | None:
    return getattr(lhotse.recipes, f'prepare_{dataset_name}', None)

def prepare_dataset_dbyml(dataset_name: str, corpus_dir: str|Path):
    fn = get_lhotse_prepare_dataset(dataset_name)
    if fn is None:
        raise ValueError("No prepare function for dataset ", dataset_name)

    return fn(corpus_dir=corpus_dir)

a = prepare_dataset_dbyml('aishell4', 'datasets/aishell4')
a

ValueError: ('No prepare function for dataset ', 'alimeeting')

In [14]:
uris = []



for recording in a['train_L']['recordings']:
    if len(recording.sources) != 1:
        raise RuntimeError(f"Invalid number of sources ({len(recording.sources)}) on recording {recording.id}")
    uris.append(recording.id)



from pyannote.core import Annotation, Segment

records: dict[str, list[tuple[Segment, str, str]]] = {uri: [] for uri in uris}
for sup_seg in a['train_L']['supervisions']:
    sup_seg_records = records[sup_seg.recording_id]
    sup_seg_records.append((Segment(sup_seg.start, sup_seg.start + sup_seg.duration), sup_seg.id, sup_seg.id))


rttms = {uri : Annotation.from_records(records=r, uri=uri) for uri, r in records.items()}

SupervisionSegment(id='20200707_L_R001S04C01-SPK0001-1', recording_id='20200707_L_R001S04C01', start=37.38507, duration=2.9, channel=[0, 1, 2, 3, 4, 5, 6, 7], text='零零二我是院长', language='Chinese', speaker='SPK0001', gender=None, custom=None, alignment=None)